Import all the libraries required.

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Load the dataset, using latin-1 encoding to interpret all the bytes. Also giving the column names because the dataset has none.

In [3]:
df = pd.read_csv(
    r"C:\Users\Mounika\Desktop\Foundations of Big Data\Assignments\training.1600000.processed.noemoticon.csv",
    encoding='latin-1',
    names=['target', 'id', 'date', 'query', 'user', 'text']
)

In [4]:
df.head()

,target,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [6]:
df.shape

(1600000, 6)

In [7]:
df.columns.tolist()

['target', 'id', 'date', 'query', 'user', 'text']

In [8]:
df.isnull().sum()

target    0
id        0
date      0
query     0
user      0
text      0
dtype: int64

In [9]:
df["target"].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df["text"].head()

0    @switchfoot http://twitpic.com/2y1zl - Awww, t...
1    is upset that he can't update his Facebook by ...
2    @Kenichan I dived many times for the ball. Man...
3      my whole body feels itchy and like its on fire 
4    @nationwideclass no, it's not behaving at all....
Name: text, dtype: str

Importing re, python's Regular expression(Regex) module for replcing, finding, searching pattern in text

In [12]:
import re

text = df["text"].astype(str)

print("Tweets containing URLs:",
      text.str.contains(r"https?://|www\.", regex=True).sum())
print("Tweets containing mentions:",
      text.str.contains(r"@\w+", regex=True).sum())
print("Tweets containing hashtags:",
      text.str.contains(r"#\w+", regex=True).sum())
print("Tweets containing digits:",
      text.str.contains(r"\d", regex=True).sum())

Tweets containing URLs: 76463
Tweets containing mentions: 738491
Tweets containing hashtags: 35842
Tweets containing digits: 366487


In [13]:
!pip install nltk

Import nltk, which is python's natural alnguage processing, for processing and analyzing human text

In [14]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Mounika\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Mounika\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mounika\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mounika\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [15]:
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

Here, I am doing preprocessing of text in two versions:

But before that, initalize tokenizer, stop words, where we are not removing negations, since they play a crucial role in determing the sentiment.


In version1:
We are just removing urls,
replacing mentions with user,
normilizing white spaces,
converting text to lower case.


In version2:
Along with version 1,
preserving hashtag word,
expanding the short forms,
applying tokenization

In [21]:
tokenizer = TweetTokenizer(preserve_case=False)
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

stop_words -= {'no', 'nor', 'not', "don't", "didn't",
               "doesn't", "isn't", "wasn't", "won't",
               "wouldn't", "shouldn't", "couldn't"}

contractions = {
    "won't": "will not",
    "can't": "cannot",
    "n't": " not",
    "i'm": "i am",
    "it's": "it is",
    "i've": "i have",
    "i'll": "i will",
    "you're": "you are",
    "we're": "we are",
    "they're": "they are",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "isn't": "is not",
    "wasn't": "was not",
    "wouldn't": "would not",
    "couldn't": "could not",
    "shouldn't": "should not"
}

def preprocess_v2(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)

    text = re.sub(r'#(\w+)', r'\1', text)

    for contraction in sorted(contractions, key=len, reverse=True):
        pattern = re.escape(contraction)
        text = re.sub(pattern, contractions[contraction], text)

    tokens = tokenizer.tokenize(text)

    text = " ".join(tokens)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [17]:
def preprocess_v1(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = text.lower()

    text = re.sub(r'@\w+', 'USER', text)

    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [18]:
df['sentiment'] = df['target'].map({
    0: 'negative',
    4: 'positive'
})

Applying both versions of processings, to comparision later on.
Creating new columns as not to loose them

In [22]:
df['clean_text'] = df['text'].apply(preprocess_v2)
df['partial_clean_text'] = df['text'].apply(preprocess_v1)

In [23]:
df[['text', 'clean_text','partial_clean_text', 'sentiment']].head()

,text,clean_text,partial_clean_text,sentiment
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...","user url - awww , that's a bummer . you should...","USER - awww, that's a bummer. you shoulda got ...",negative
1,is upset that he can't update his Facebook by ...,is upset that he cannot update his facebook by...,is upset that he can't update his facebook by ...,negative
2,@Kenichan I dived many times for the ball. Man...,user i dived many times for the ball . managed...,USER i dived many times for the ball. managed ...,negative
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,negative
4,"@nationwideclass no, it's not behaving at all....","user no , it is not behaving at all . i am mad...","USER no, it's not behaving at all. i'm mad. wh...",negative


Saving the preprocessed data to a csv file

In [18]:
processed_df = df[['target', 'text', 'clean_text', 'partial_clean_text', 'sentiment']]

# Save to a separate CSV file
processed_df.to_csv(
    'sentimentalAnalysis_preprocessed.csv',
    index=False,
    encoding='utf-8'
)

print("Preprocessed dataset saved successfully!")

Preprocessed dataset saved successfully!
